# Task 2 — Sentiment Analysis: Fine-tuning a Pre-trained Transformer

**Goal:** Build a 3-class sentiment classifier (`positive` / `neutral` / `negative`) by fine-tuning a pre-trained transformer on the Kaggle [Sentiment Analysis Dataset](https://www.kaggle.com/datasets/abhi8923shriv/sentiment-analysis-dataset).

**Approach**
1. Pull the dataset straight from Kaggle via `kagglehub` (no manual download needed).
2. Do a short EDA to understand class balance and text length.
3. Clean tweets lightly (URLs, mentions, HTML entities).
4. Tokenise with the **DistilBERT** tokenizer (`distilbert-base-uncased`) — DistilBERT is ~40% smaller than BERT but retains ~97% of its performance, making it ideal for a Colab T4.
5. Fine-tune `DistilBertForSequenceClassification` with:
   - **Class-weighted** cross-entropy to handle imbalance.
   - **AdamW** + **cosine LR schedule** with warmup.
   - **Mixed precision** training for speed.
   - **Early stopping** on validation loss.
6. Report classification metrics, confusion matrix, and per-class ROC-AUC on the test set.

> **Runtime:** Google Colab (GPU). `Runtime → Change runtime type → T4 GPU` before running.


## Step 1 — Install dependencies

In [ ]:
!pip install -q transformers datasets kagglehub

## Step 2 — Imports & reproducibility

In [ ]:
import os, re, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    accuracy_score, f1_score, precision_score, recall_score,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

## Step 3 — Download the dataset from Kaggle

`kagglehub` handles authentication and caching automatically. The first time you run this in a fresh Colab session it will prompt you for your Kaggle username/key (get them from kaggle.com → Account → Create New API Token).

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "abhi8923shriv/sentiment-analysis-dataset"
)
print("Dataset downloaded to:", dataset_path)

# List what's inside — the dataset contains train.csv and test.csv
for p in sorted(Path(dataset_path).iterdir()):
    print(" ", p.name)

In [ ]:
# Load the CSVs (latin-1 encoding because the files contain non-UTF-8 chars)
TRAIN_CSV = Path(dataset_path) / "train.csv"
TEST_CSV  = Path(dataset_path) / "test.csv"

train_df = pd.read_csv(TRAIN_CSV, encoding="latin1")
test_df  = pd.read_csv(TEST_CSV,  encoding="latin1")

print(f"Train: {train_df.shape}")
print(f"Test : {test_df.shape}")
print()
print("Columns:", list(train_df.columns))
train_df.head(3)

## Step 4 — Exploratory data analysis

In [ ]:
# Missing values
print("Missing values in train:")
print(train_df[["text", "sentiment"]].isnull().sum())

# Drop rows with missing sentiment or text
train_df = train_df.dropna(subset=["text", "sentiment"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["text", "sentiment"]).reset_index(drop=True)

# Class distribution
label_counts = train_df["sentiment"].value_counts()
print(f"\nClass distribution:\n{label_counts}")

In [ ]:
# Visual EDA: class balance + text length distribution
colors = {"positive": "#2ecc71", "neutral": "#3498db", "negative": "#e74c3c"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("EDA — training set", fontweight="bold")

# (1) Bar chart
bars = axes[0].bar(label_counts.index, label_counts.values,
                   color=[colors[c] for c in label_counts.index], edgecolor="k")
axes[0].bar_label(bars, fontsize=10, fontweight="bold")
axes[0].set_title("Class distribution")
axes[0].set_ylabel("Count"); axes[0].grid(axis="y", alpha=0.3)

# (2) Pie chart
axes[1].pie(label_counts.values, labels=label_counts.index,
            colors=[colors[c] for c in label_counts.index],
            autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white"})
axes[1].set_title("Class proportions")

# (3) Word-length distribution by class
train_df["word_len"] = train_df["text"].astype(str).apply(lambda t: len(t.split()))
for cls, col in colors.items():
    sub = train_df[train_df["sentiment"] == cls]["word_len"]
    axes[2].hist(sub, bins=30, alpha=0.55, label=cls, color=col, edgecolor="k")
axes[2].set_title("Word count per tweet")
axes[2].set_xlabel("Words"); axes[2].set_ylabel("Frequency")
axes[2].legend(); axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout(); plt.show()

print(f"\nWord-length stats: mean={train_df['word_len'].mean():.1f}, "
      f"p95={train_df['word_len'].quantile(0.95):.0f}, "
      f"max={train_df['word_len'].max()}")

## Step 5 — Clean text & encode labels

We apply only light cleaning — DistilBERT is surprisingly good at handling casing and punctuation, so we don't strip them. We do remove URLs, `@mentions`, and HTML entities because they carry no sentiment signal and just consume token budget.

In [ ]:
LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
CLASS_NAMES = ["negative", "neutral", "positive"]
NUM_LABELS = 3


def clean_text(t: str) -> str:
    t = str(t)
    t = re.sub(r"http\S+|www\.\S+", "", t)   # URLs
    t = re.sub(r"@\w+", "@user", t)            # normalise mentions
    t = re.sub(r"#", "", t)                      # keep hashtag word, drop #
    t = re.sub(r"&amp;", "&", t)                 # decode HTML entity
    t = re.sub(r"\s+", " ", t).strip()
    return t


for df in (train_df, test_df):
    df["clean_text"] = df["text"].apply(clean_text)
    df["sentiment"]  = df["sentiment"].str.strip().str.lower()
    df["label"]      = df["sentiment"].map(LABEL_MAP)

# Drop rows whose sentiment isn't one of our 3 classes (shouldn't be any)
train_df = train_df.dropna(subset=["label"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["label"]).reset_index(drop=True)
train_df["label"] = train_df["label"].astype(int)
test_df["label"]  = test_df["label"].astype(int)

print(f"After cleaning — train: {len(train_df)}, test: {len(test_df)}")

### Split train into train + val (90 / 10)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df, test_size=0.1, stratify=train_df["label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## Step 6 — Tokeniser & PyTorch Dataset

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 96   # 95th percentile of tweet lengths is well under this

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {MODEL_NAME}  (vocab size: {tokenizer.vocab_size:,})")

# Double-check token lengths
token_lens = train_df["clean_text"].apply(
    lambda t: len(tokenizer.encode(t, add_special_tokens=True))
)
print(f"Token length: mean={token_lens.mean():.1f}  "
      f"p95={token_lens.quantile(0.95):.0f}  max={token_lens.max()}")

In [ ]:
class SentimentDataset(Dataset):
    """Tokenise on the fly so we can keep raw text around for error analysis."""
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[i], dtype=torch.long),
            "text":           self.texts[i],
        }


BATCH_SIZE = 32

train_ds = SentimentDataset(train_df["clean_text"], train_df["label"], tokenizer, MAX_LEN)
val_ds   = SentimentDataset(val_df["clean_text"],   val_df["label"],   tokenizer, MAX_LEN)
test_ds  = SentimentDataset(test_df["clean_text"],  test_df["label"],  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

## Step 7 — Compute class weights for imbalanced loss

The dataset is imbalanced (neutral is typically the most frequent class). Using inverse-frequency class weights in `CrossEntropyLoss` stops the model from just predicting the majority class to minimise loss.

In [ ]:
counts = np.bincount(train_df["label"], minlength=NUM_LABELS).astype(float)
class_weights = torch.tensor(
    len(train_df) / (NUM_LABELS * counts), dtype=torch.float32
).to(DEVICE)

for i, cls in enumerate(CLASS_NAMES):
    print(f"  {cls:<9}  count={int(counts[i]):>6}   weight={class_weights[i].item():.4f}")

## Step 8 — Load pre-trained DistilBERT for sequence classification

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model     : {MODEL_NAME}")
print(f"Parameters: {total:,} total ({train:,} trainable)")

## Step 9 — Optimiser, scheduler & loss

**Layer-wise LR decay**: lower transformer layers (closer to the input) already contain great general-purpose features, so we train them with a smaller LR than the higher layers and the classification head. This is a well-established trick that improves transformer fine-tuning stability.

**Cosine schedule with warmup**: the LR ramps up linearly for the first 10% of steps, then decays as a cosine. Warmup is important for transformers because AdamW's moment estimates are noisy early on.

In [ ]:
NUM_EPOCHS    = 4
BASE_LR       = 3e-5        # LR for the classifier head
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
GRAD_CLIP     = 1.0
LAYER_LR_DECAY = 0.9
PATIENCE      = 2

no_decay = ("bias", "LayerNorm.weight")
param_groups = []

# Classifier head (pre_classifier + classifier in DistilBERT) — full LR
head_modules = [("pre_classifier", model.pre_classifier),
                ("classifier",     model.classifier)]
for _, mod in head_modules:
    param_groups.append({
        "params": [p for n, p in mod.named_parameters() if not any(nd in n for nd in no_decay)],
        "lr": BASE_LR, "weight_decay": WEIGHT_DECAY,
    })
    param_groups.append({
        "params": [p for n, p in mod.named_parameters() if     any(nd in n for nd in no_decay)],
        "lr": BASE_LR, "weight_decay": 0.0,
    })

# Transformer encoder layers — progressively smaller LRs
encoder_layers = model.distilbert.transformer.layer
num_layers     = len(encoder_layers)
for i, layer in enumerate(encoder_layers):
    # Layer closer to input (i=0) → smallest LR
    layer_lr = BASE_LR * (LAYER_LR_DECAY ** (num_layers - 1 - i))
    param_groups.append({
        "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
        "lr": layer_lr, "weight_decay": WEIGHT_DECAY,
    })
    param_groups.append({
        "params": [p for n, p in layer.named_parameters() if     any(nd in n for nd in no_decay)],
        "lr": layer_lr, "weight_decay": 0.0,
    })

# Embeddings — smallest LR
emb_lr = BASE_LR * (LAYER_LR_DECAY ** num_layers)
param_groups.append({
    "params": [p for n, p in model.distilbert.embeddings.named_parameters()
               if not any(nd in n for nd in no_decay)],
    "lr": emb_lr, "weight_decay": WEIGHT_DECAY,
})
param_groups.append({
    "params": [p for n, p in model.distilbert.embeddings.named_parameters()
               if     any(nd in n for nd in no_decay)],
    "lr": emb_lr, "weight_decay": 0.0,
})

optimizer = torch.optim.AdamW(param_groups)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler    = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler    = GradScaler()

print(f"Total steps : {total_steps}")
print(f"Warmup      : {warmup_steps}")
print(f"LR range    : {emb_lr:.2e} (embeddings) → {BASE_LR:.2e} (head)")

## Step 10 — Training & evaluation helpers

In [ ]:
def train_one_epoch():
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in train_loader:
        ids, mask, lbls = (batch["input_ids"].to(DEVICE),
                           batch["attention_mask"].to(DEVICE),
                           batch["label"].to(DEVICE))

        optimizer.zero_grad()
        with autocast():
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = criterion(logits, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        scheduler.step()

        total_loss += loss.item() * lbls.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        total      += lbls.size(0)
    return total_loss / total, 100 * correct / total


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    y_true, y_pred, y_prob, texts = [], [], [], []
    for batch in loader:
        ids, mask, lbls = (batch["input_ids"].to(DEVICE),
                           batch["attention_mask"].to(DEVICE),
                           batch["label"].to(DEVICE))
        logits = model(input_ids=ids, attention_mask=mask).logits
        loss   = criterion(logits, lbls)

        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(1)

        total_loss += loss.item() * lbls.size(0)
        correct    += (preds == lbls.cpu().numpy()).sum()
        total      += lbls.size(0)

        y_true.extend(lbls.cpu().numpy()); y_pred.extend(preds); y_prob.extend(probs)
        texts.extend(batch["text"])
    return (total_loss / total, 100 * correct / total,
            np.array(y_true), np.array(y_pred), np.array(y_prob), texts)

## Step 11 — Fine-tuning loop with early stopping

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

best_val_loss  = float("inf")
best_weights   = None
patience_count = 0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch()
    va_loss, va_acc, *_ = evaluate(val_loader)

    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss);   history["val_acc"].append(va_acc)

    tag = ""
    if va_loss < best_val_loss:
        best_val_loss  = va_loss
        best_weights   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_count = 0
        tag = "  ← new best"
    else:
        patience_count += 1

    print(f"Epoch {epoch}/{NUM_EPOCHS}  "
          f"train loss {tr_loss:.4f} acc {tr_acc:5.2f}%  |  "
          f"val loss {va_loss:.4f} acc {va_acc:5.2f}%{tag}")

    if patience_count >= PATIENCE:
        print(f"Early stopping (no improvement for {PATIENCE} epochs).")
        break

# Restore best weights
model.load_state_dict(best_weights)
print(f"\nBest val loss: {best_val_loss:.4f}")

## Step 12 — Training curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["val_loss"],   "s-", label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["train_acc"], "o-", label="train")
axes[1].plot(epochs, history["val_acc"],   "s-", label="val")
axes[1].set_title("Accuracy (%)"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Step 13 — Classification report on the test set

In [ ]:
_, test_acc, y_true, y_pred, y_prob, test_texts = evaluate(test_loader)

print("=" * 60)
print("CLASSIFICATION REPORT — Test Set")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

print(f"Test accuracy: {test_acc:.2f}%")

## Step 14 — Confusion matrix & per-class ROC-AUC

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Confusion matrix (counts)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Oranges",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Confusion matrix (row-normalised = recall)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

plt.tight_layout(); plt.show()

In [ ]:
# One-vs-rest ROC curves
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
class_colors = ["#e74c3c", "#3498db", "#2ecc71"]

fig, ax = plt.subplots(figsize=(7, 6))
auc_scores = {}
for i, (cls, col) in enumerate(zip(CLASS_NAMES, class_colors)):
    if len(np.unique(y_true_bin[:, i])) < 2:
        continue
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores[cls] = roc_auc
    ax.plot(fpr, tpr, color=col, lw=2, label=f"{cls} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curves — one-vs-rest")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

for cls, s in auc_scores.items():
    print(f"  AUC-ROC ({cls}): {s:.4f}")

## Step 15 — Sample misclassifications

In [ ]:
errors = np.where(y_pred != y_true)[0]
print(f"Misclassified: {len(errors)} / {len(y_true)}  "
      f"({100 * len(errors) / len(y_true):.1f}%)")

print("\nFirst 10 errors (highest-confidence wrong predictions are often the most interesting):")
# Sort errors by confidence in the wrong prediction (descending)
err_conf = [(idx, y_prob[idx, y_pred[idx]]) for idx in errors]
err_conf.sort(key=lambda x: -x[1])

for idx, conf in err_conf[:10]:
    print(f"  True: {CLASS_NAMES[y_true[idx]]:<9}  "
          f"Pred: {CLASS_NAMES[y_pred[idx]]:<9}  "
          f"Conf: {conf:.3f}")
    text = test_texts[idx][:100]
    print(f"  Text: \"{text}\"\n")

## Step 16 — Final summary

In [ ]:
acc  = accuracy_score(y_true, y_pred) * 100
prec = precision_score(y_true, y_pred, average="weighted", zero_division=0) * 100
rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0) * 100
f1_w = f1_score(y_true, y_pred, average="weighted", zero_division=0) * 100
f1_m = f1_score(y_true, y_pred, average="macro",    zero_division=0) * 100

print("=" * 60)
print("  FINAL RESULTS — Sentiment Analysis")
print("=" * 60)
print(f"  Model              : {MODEL_NAME}")
print(f"  Epochs completed   : {len(history['train_loss'])}")
print(f"  Max sequence length: {MAX_LEN}")
print(f"  Batch size         : {BATCH_SIZE}")
print(f"  Class weighting    : inverse frequency")
print()
print(f"  Test accuracy      : {acc:.2f}%")
print(f"  Weighted precision : {prec:.2f}%")
print(f"  Weighted recall    : {rec:.2f}%")
print(f"  Weighted F1        : {f1_w:.2f}%")
print(f"  Macro F1           : {f1_m:.2f}%")
for cls, s in auc_scores.items():
    print(f"  AUC-ROC ({cls:<8}): {s:.4f}")
print("=" * 60)

print("\nFull classification report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

## Step 17 — Try it on your own sentences

In [ ]:
@torch.no_grad()
def predict_sentiment(text: str):
    model.eval()
    enc = tokenizer(clean_text(text),
                    max_length=MAX_LEN, padding="max_length",
                    truncation=True, return_tensors="pt")
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)
    probs = torch.softmax(model(input_ids=ids, attention_mask=mask).logits,
                          dim=1)[0].cpu().numpy()
    pred = probs.argmax()
    return CLASS_NAMES[pred], float(probs[pred]), \
           {CLASS_NAMES[i]: float(p) for i, p in enumerate(probs)}


demos = [
    "I absolutely love this! Best day ever :)",
    "The package arrived today. It's okay I guess.",
    "Terrible service, never coming back. Ruined my evening.",
    "Just had coffee and reading the news. Nothing special.",
    "@user your support team was AMAZING, fixed my issue in 2 mins!",
]

print("=" * 55)
print("  LIVE INFERENCE")
print("=" * 55)
for text in demos:
    pred, conf, probs = predict_sentiment(text)
    print(f"\n\"{text}\"")
    print(f"  → {pred.upper()}  (confidence: {conf:.3f})")
    for cls, p in probs.items():
        bar = "█" * int(p * 25)
        print(f"    {cls:<9} {p:.3f}  {bar}")